# CIC-IDS2017: стратифікований часовий спліт (виправлені таймстемпи)

**Навіщо.** Хронологічний спліт 70/85 некоректний для CIC-IDS2017, бо кожного дня записували інший тип атак. Через це в test потрапляють класи, яких немає в train, а val не відповідає test.

**Дизайн.** Таймстемпи виправлено (AM/PM, +12 год). Групи = (день × клас). У кожній групі потоки впорядковано за часом: перші 70% → train, наступні 15% → val, останні 15% → test. Межі визначаються так само, як у замороженому протоколі: `quantile(..., interpolation="nearest")`, train = t < b1, val = b1 ≤ t < b2, test = t ≥ b2.

**Обмеження дизайну (вказати в статті).** Спліт хронологічний лише *всередині* кожної групи. Train містить трафік пізніших днів, ніж test ранніх днів. Отже, це оцінка «у межах розподілу», а не прогноз у майбутнє.

## Гіпотеза і критерії (записано ДО запуску)
- **Первинна H1:** при 500 деревах (верхня межа `num_boost_round` замороженого протоколу, без підбору) ΔAP(F1−F0) на test > 0, і нижня межа 95% блочного bootstrap CI (блоки = хвилини) > 0.
- **Вторинні (звітуються всі, незалежно від результату):**
  - (a) криві ΔAP при 50/100/200/300/500 деревах;
  - (b) порівняння з early stopping на val — лише як довідка;
  - (c) контроль порту: F0 + destination_port проти F1 при 500 деревах;
  - (d) контроль змісту CTI: F0 + числові CTI-ознаки (справжні) проти F0 + ті самі ознаки, переставлені між сервісами (групування збережене, значення чужі), при 500 деревах;
  - (e) AP «клас проти BENIGN» по класах при 500 деревах. Класи з механізмом CVE (Heartbleed, Infiltration, Web Attack*) мають дуже малі n, тому подаються лише описово.

In [1]:
import pandas as pd, numpy as np, lightgbm as lgb, json, os
from sklearn.metrics import average_precision_score, roc_auc_score

PATH = "/srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet"
TS_COL, LABEL_COL, PORT_COL = "Timestamp", "attack_label", "destination_port"
SEED = 42
ROUNDS = 500

F0_CIC = ['Flow Duration','Total Fwd Packets','Total Backward Packets','Total Length of Fwd Packets',
'Total Length of Bwd Packets','Fwd Packet Length Max','Fwd Packet Length Min','Fwd Packet Length Mean',
'Fwd Packet Length Std','Bwd Packet Length Max','Bwd Packet Length Min','Bwd Packet Length Mean',
'Bwd Packet Length Std','Flow Bytes/s','Flow Packets/s','Flow IAT Mean','Flow IAT Std','Flow IAT Max',
'Flow IAT Min','Fwd IAT Total','Fwd IAT Mean','Fwd IAT Std','Fwd IAT Max','Fwd IAT Min','Bwd IAT Total',
'Bwd IAT Mean','Bwd IAT Std','Bwd IAT Max','Bwd IAT Min','Fwd PSH Flags','Bwd PSH Flags','Fwd URG Flags',
'Bwd URG Flags','Fwd Header Length','Bwd Header Length','Fwd Packets/s','Bwd Packets/s','Min Packet Length',
'Max Packet Length','Packet Length Mean','Packet Length Std','Packet Length Variance','FIN Flag Count',
'SYN Flag Count','RST Flag Count','PSH Flag Count','ACK Flag Count','URG Flag Count','CWE Flag Count',
'ECE Flag Count','Down/Up Ratio','Average Packet Size','Avg Fwd Segment Size','Avg Bwd Segment Size',
'Fwd Avg Bytes/Bulk','Fwd Avg Packets/Bulk','Fwd Avg Bulk Rate','Bwd Avg Bytes/Bulk','Bwd Avg Packets/Bulk',
'Bwd Avg Bulk Rate','Subflow Fwd Packets','Subflow Fwd Bytes','Subflow Bwd Packets','Subflow Bwd Bytes',
'Init_Win_bytes_forward','Init_Win_bytes_backward','act_data_pkt_fwd','min_seg_size_forward','Active Mean',
'Active Std','Active Max','Active Min','Idle Mean','Idle Std','Idle Max','Idle Min','protocol']
CVE_CTX = ['cve_ctx_service','cve_ctx_pool_size','cve_ctx_n_epss','cve_ctx_epss_coverage',
           'cve_ctx_service_max_epss','cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']
NUM_CTX = [c for c in CVE_CTX if c != 'cve_ctx_service']
F1_CIC = F0_CIC + CVE_CTX
CAT_COLS = {"protocol", "cve_ctx_service"}

df = pd.read_parquet(PATH)
df.columns = [c.strip() for c in df.columns]
print("rows:", len(df), "| відсутні колонки:",
      [c for c in F1_CIC + [PORT_COL, TS_COL, LABEL_COL] if c not in df.columns] or "— жодної")
y = (df[LABEL_COL].astype(str) != "BENIGN").astype(int)

rows: 2827677 | відсутні колонки: — жодної


In [2]:
# 1. Виправлення AM/PM і стратифікований часовий спліт (група = день × клас)
raw = df[TS_COL]
ts = pd.to_datetime(raw) if np.issubdtype(raw.dtype, np.datetime64) else pd.to_datetime(raw.astype(str), dayfirst=True, errors="coerce")
ts = ts.where(ts.dt.hour >= 8, ts + pd.Timedelta(hours=12))
df["_ts"] = ts
grp = ts.dt.date.astype(str) + "|" + df[LABEL_COL].astype(str)
tnum = ts.astype("int64")
b1 = tnum.groupby(grp).transform(lambda s: s.quantile(0.70, interpolation="nearest"))
b2 = tnum.groupby(grp).transform(lambda s: s.quantile(0.85, interpolation="nearest"))
df["_split"] = np.where(tnum < b1, "train", np.where(tnum < b2, "val", "test"))
tr = (df["_split"] == "train").to_numpy(dtype=bool)
va = (df["_split"] == "val").to_numpy(dtype=bool)
te = (df["_split"] == "test").to_numpy(dtype=bool)

ct = pd.crosstab(df[LABEL_COL], df["_split"])[["train", "val", "test"]]
print(ct.to_string())
print("\nчастки (train/val/test):", np.round(np.array([tr.sum(), va.sum(), te.sum()]) / len(df), 4))
print("частка атак:", {k: round(float(y.to_numpy()[m].mean()) * 100, 2) for k, m in [("train", tr), ("val", va), ("test", te)]})
print("AP випадкового класифікатора на test:", round(float(y.to_numpy()[te].mean()), 6))
print("\nКласи, відсутні в train, але присутні в test:",
      [c for c in ct.index if ct.loc[c, "train"] == 0 and ct.loc[c, "test"] > 0] or "— немає")

_split                        train     val    test
attack_label                                       
BENIGN                      1587637  341811  341674
Bot                            1330     332     294
DDoS                          88459   19116   20450
DoS GoldenEye                  6170    1330    2793
DoS Hulk                     158576   23989   47558
DoS Slowhttptest               3162    1133    1204
DoS slowloris                  3933     943     920
FTP-Patator                    5517    1184    1234
Heartbleed                        7       1       3
Infiltration                     24       6       6
PortScan                      93404       0   65400
SSH-Patator                    4123     870     904
Web Attack – Brute Force       1032     233     242
Web Attack – Sql Injection        8       8       5
Web Attack – XSS                434      99     119

частки (train/val/test): [0.691  0.1383 0.1707]
частка атак: {'train': 18.74, 'val': 12.59, 'test': 29.23}
AP випад

In [3]:
# 2. Допоміжні функції (заморожені гіперпараметри, Table 5)
y_np = y.to_numpy()
SPW = (y_np[tr] == 0).sum() / y_np[tr].sum()
PARAMS = dict(objective="binary", metric="None", seed=SEED, num_leaves=15, learning_rate=0.05,
              min_data_in_leaf=50, verbosity=-1, scale_pos_weight=SPW)
print("scale_pos_weight:", round(float(SPW), 4))

def prep(frame_src, cols):
    frame = frame_src[cols].copy()
    cats = [c for c in CAT_COLS if c in cols]
    for c in cats:
        frame[c] = pd.Categorical(frame[c], categories=pd.Categorical(frame.loc[tr, c]).categories)
    return frame, cats

def fit_fixed(frame_src, cols, tag):
    frame, cats = prep(frame_src, cols)
    m = lgb.train(PARAMS, lgb.Dataset(frame[tr], label=y_np[tr], categorical_feature=cats), num_boost_round=ROUNDS)
    print(f"[{tag}] навчено {ROUNDS} дерев, {len(cols)} ознак")
    return m, frame

def ap_at(model, frame, r):
    return average_precision_score(y_np[te], model.predict(frame[te], num_iteration=r))

def ap_feval(preds, data):
    return "unweighted_ap", average_precision_score(data.get_label(), preds), True

scale_pos_weight: 4.3357


In [4]:
# 3. Моделі з фіксованим бюджетом (одне навчання на 500 дерев, оцінка на проміжних бюджетах)
models = {}
models["F0"] = fit_fixed(df, F0_CIC, "F0")
models["F1"] = fit_fixed(df, F1_CIC, "F1")
models["F0+port"] = fit_fixed(df, F0_CIC + [PORT_COL], "F0 + destination_port")
models["F0+num_real"] = fit_fixed(df, F0_CIC + NUM_CTX, "F0 + числові CTI (справжні)")

# числові CTI-ознаки, переставлені між сервісами (групування збережене, значення від іншого сервісу)
key = df["cve_ctx_service"].astype(str)
svc_stats = df.groupby(key, observed=True)[NUM_CTX].first()
svc_stats = svc_stats[~svc_stats.index.isin(["nan", "<NA>", "None"])]
perm = np.random.default_rng(SEED).permutation(len(svc_stats))
df_perm = df.copy()
for c in NUM_CTX:
    mapping = dict(zip(svc_stats.index, svc_stats[c].to_numpy()[perm]))
    df_perm[c] = pd.to_numeric(key.map(mapping).where(key.isin(list(mapping.keys())), df[c]), errors="coerce")
models["F0+num_perm"] = fit_fixed(df_perm, F0_CIC + NUM_CTX, "F0 + числові CTI (переставлені між сервісами)")

[F0] навчено 500 дерев, 77 ознак
[F1] навчено 500 дерев, 84 ознак
[F0 + destination_port] навчено 500 дерев, 78 ознак
[F0 + числові CTI (справжні)] навчено 500 дерев, 83 ознак
[F0 + числові CTI (переставлені між сервісами)] навчено 500 дерев, 83 ознак


In [5]:
# 4. Вторинне (a): криві ΔAP при однаковому бюджеті
rows = []
for r in [50, 100, 200, 300, 500]:
    a = {k: ap_at(m, f, r) for k, (m, f) in models.items()}
    rows.append({"rounds": r, **{f"AP_{k}": v for k, v in a.items()}, "dAP_F1_F0": a["F1"] - a["F0"]})
    print(f"rounds={r:>3}  F0={a['F0']:.6f}  F1={a['F1']:.6f}  dAP={a['F1'] - a['F0']:+.6f}  "
          f"F0+port={a['F0+port']:.6f}  num_real={a['F0+num_real']:.6f}  num_perm={a['F0+num_perm']:.6f}")
curve = pd.DataFrame(rows)

rounds= 50  F0=0.997753  F1=0.998754  dAP=+0.001002  F0+port=0.999062  num_real=0.998791  num_perm=0.998903
rounds=100  F0=0.998494  F1=0.999410  dAP=+0.000916  F0+port=0.999520  num_real=0.999366  num_perm=0.999499
rounds=200  F0=0.998673  F1=0.999560  dAP=+0.000886  F0+port=0.999534  num_real=0.999388  num_perm=0.999506
rounds=300  F0=0.998793  F1=0.999607  dAP=+0.000814  F0+port=0.999598  num_real=0.999544  num_perm=0.999519
rounds=500  F0=0.998869  F1=0.999402  dAP=+0.000533  F0+port=0.999489  num_real=0.999473  num_perm=0.999538


In [6]:
# 5. ПЕРВИННА H1: ΔAP(F1−F0) при 500 деревах + блочний bootstrap CI (блоки = хвилини)
N_BOOT = 500
p = {k: m.predict(f[te], num_iteration=ROUNDS) for k, (m, f) in models.items()}
y_te = y_np[te]
blk = pd.factorize(df.loc[te, "_ts"].dt.floor("min"))[0]
blocks = list(pd.Series(np.arange(len(blk))).groupby(blk).indices.values())
print("рядків у test:", len(y_te), "| хвилинних блоків:", len(blocks))
rng = np.random.default_rng(SEED)

def block_ci(pa, pb):
    point = average_precision_score(y_te, pa) - average_precision_score(y_te, pb)
    vals = []
    for _ in range(N_BOOT):
        j = np.concatenate([blocks[k] for k in rng.integers(0, len(blocks), len(blocks))])
        vals.append(average_precision_score(y_te[j], pa[j]) - average_precision_score(y_te[j], pb[j]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return point, lo, hi, float(np.mean(np.array(vals) > 0))

res = []
for name, a, b in [("H1: F1 − F0", "F1", "F0"),
                   ("(c) F1 − F0+port", "F1", "F0+port"),
                   ("(c) F0+port − F0", "F0+port", "F0"),
                   ("(d) num_real − num_perm", "F0+num_real", "F0+num_perm")]:
    pt, lo, hi, share = block_ci(p[a], p[b])
    res.append({"comparison": name, "dAP": pt, "CI95_lo": lo, "CI95_hi": hi, "P(dAP>0)": share})
    print(f"{name:<26} dAP={pt:+.6f}  95% block-CI [{lo:+.6f}, {hi:+.6f}]  частка реплік >0: {share:.3f}")
ci_df = pd.DataFrame(res)
h1 = ci_df.iloc[0]
print("\nH1 ПІДТВЕРДЖЕНА" if (h1.dAP > 0 and h1.CI95_lo > 0) else "\nH1 НЕ підтверджена")

рядків у test: 482806 | хвилинних блоків: 511
H1: F1 − F0                dAP=+0.000533  95% block-CI [+0.000183, +0.001988]  частка реплік >0: 1.000
(c) F1 − F0+port           dAP=-0.000086  95% block-CI [-0.000309, +0.000041]  частка реплік >0: 0.112
(c) F0+port − F0           dAP=+0.000619  95% block-CI [+0.000230, +0.002030]  частка реплік >0: 1.000
(d) num_real − num_perm    dAP=-0.000065  95% block-CI [-0.000270, -0.000001]  частка реплік >0: 0.024

H1 ПІДТВЕРДЖЕНА


In [7]:
# 6. Вторинне (b): early stopping на val — лише довідка (val тепер репрезентативний)
for k, cols in [("F0", F0_CIC), ("F1", F1_CIC)]:
    frame, cats = prep(df, cols)
    dtr = lgb.Dataset(frame[tr], label=y_np[tr], categorical_feature=cats, free_raw_data=False)
    dva = lgb.Dataset(frame[va], label=y_np[va], categorical_feature=cats, reference=dtr, free_raw_data=False)
    m = lgb.train(PARAMS, dtr, num_boost_round=ROUNDS, valid_sets=[dva], feval=ap_feval,
                  callbacks=[lgb.early_stopping(30, verbose=False)])
    pr = m.predict(frame[te], num_iteration=m.best_iteration)
    print(f"[{k} early-stop] best_iter={m.best_iteration} test_AP={average_precision_score(y_te, pr):.6f} "
          f"test_AUC={roc_auc_score(y_te, pr):.6f}")

[F0 early-stop] best_iter=66 test_AP=0.998154 test_AUC=0.998915
[F1 early-stop] best_iter=115 test_AP=0.999492 test_AUC=0.999749


In [8]:
# 7. Вторинне (e): AP «клас проти BENIGN» при 500 деревах
lab_te = df.loc[te, LABEL_COL].astype(str).to_numpy()
benign = lab_te == "BENIGN"
CVE_MECH = {"Heartbleed", "Infiltration"} | {c for c in set(lab_te) if c.startswith("Web Attack")}
rows = []
for cls in sorted(set(lab_te) - {"BENIGN"}):
    pos = lab_te == cls
    msk = pos | benign
    yy = pos[msk].astype(int)
    a0, a1 = average_precision_score(yy, p["F0"][msk]), average_precision_score(yy, p["F1"][msk])
    rows.append({"attack_label": cls, "cve_mechanism": cls in CVE_MECH, "n_test": int(pos.sum()),
                 "AP_F0": a0, "AP_F1": a1, "dAP": a1 - a0,
                 "err_reduction_%": 100 * (a1 - a0) / (1 - a0) if a0 < 1 else np.nan})
per_cls = pd.DataFrame(rows).sort_values(["cve_mechanism", "n_test"], ascending=[False, False])
print(per_cls.round(6).to_string(index=False))

              attack_label  cve_mechanism  n_test    AP_F0    AP_F1       dAP  err_reduction_%
  Web Attack – Brute Force           True     242 0.933652 0.983683  0.050032        75.407457
          Web Attack – XSS           True     119 0.906840 0.920701  0.013861        14.878926
              Infiltration           True       6 0.001035 0.027313  0.026278         2.630563
Web Attack – Sql Injection           True       5 0.001146 0.013763  0.012617         1.263134
                Heartbleed           True       3 0.000052 0.000253  0.000201         0.020061
                  PortScan          False   65400 0.999911 0.999798 -0.000112      -125.632242
                  DoS Hulk          False   47558 0.997177 0.998654  0.001476        52.302771
                      DDoS          False   20450 0.998816 0.999404  0.000588        49.671933
             DoS GoldenEye          False    2793 0.985077 0.992322  0.007244        48.545851
               FTP-Patator          False    1234 

In [9]:
# 8. Збереження
out = "/srv/ids_research/data"
curve.to_csv(f"{out}/cic_strat_curve.csv", index=False)
ci_df.to_csv(f"{out}/cic_strat_bootstrap_ci.csv", index=False)
per_cls.to_csv(f"{out}/cic_strat_per_class.csv", index=False)
ct.to_csv(f"{out}/cic_strat_split_composition.csv")
print("Збережено: cic_strat_curve.csv, cic_strat_bootstrap_ci.csv, cic_strat_per_class.csv, cic_strat_split_composition.csv")

Збережено: cic_strat_curve.csv, cic_strat_bootstrap_ci.csv, cic_strat_per_class.csv, cic_strat_split_composition.csv
